# tcpyVPI: ERA5 vPI and GPIv, installed **from GitHub**

This notebook is the same workflow as the other ERA5 examples, but it installs
`tcpyVPI` **directly from the GitHub repository** rather than from PyPI. Use this
when you want the current state of `main` (or a specific tag) rather than waiting
for a PyPI release.

IMPORTANT: this notebook reads ERA5 data remotely via the NCAR THREDDS server.
Sometimes THREDDS throws a `NetCDF: DAP server error` when reading the data -- it
seems to be somewhat random whether/when it happens and is not due to this
notebook. If it happens, run it again.

**Cite this package:**  
Chavas, D. Sanchez, J. O., and A. Kruskie (2026). *tcpyVPI*. https://doi.org/10.5281/zenodo.19319996

[![PyPI version](https://img.shields.io/pypi/v/tcpyVPI.svg)](https://pypi.org/project/tcpyVPI/)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19319996.svg)](https://doi.org/10.5281/zenodo.19319996)

**Author:** Dan Chavas (2026)


## 0. Install from GitHub

`pip` installs straight from the repo with the `git+https://` scheme. Tracking
`main` gives you the newest code immediately. Pinning a tag gives reproducibility,
but note a tag only exists once the corresponding GitHub Release is published.


In [ ]:
# Track the latest commit on main -- works right now
# !pip install -q "git+https://github.com/drchavas/tcpyVPI.git@main"

# Once the vX.Y.Z Release is published on GitHub, pin the tag instead for
# reproducibility (a tag only exists after you publish the Release):
!pip install -q "git+https://github.com/drchavas/tcpyVPI.git@v1.0.1"

# Force a reinstall if you already have tcpyVPI from PyPI in this runtime,
# or if you are re-running after pushing new commits (pip caches by URL):
# !pip install -q --force-reinstall --no-deps "git+https://github.com/drchavas/tcpyVPI.git@main"

!pip install -q tcpyPI cartopy

### Confirm it really came from GitHub, not PyPI

A PyPI install shows a bare version (`tcpyVPI==1.1.0`); a git install shows the
repository URL and the exact commit it was built from.


In [ ]:
import subprocess, tcpyVPI

print("version :", tcpyVPI.__version__)
print("path    :", tcpyVPI.__file__)
print()
freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
line = [l for l in freeze.splitlines() if l.lower().startswith("tcpyvpi")]
print("pip freeze:", line[0] if line else "(not found)")
print()
if line and "git+" in line[0]:
    print("OK - installed from GitHub")
else:
    print("WARNING - this looks like the PyPI build, not the GitHub one.")
    print("Re-run the install cell with --force-reinstall.")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from tcpyVPI import (
    run_vpigpiv,
    load_era5_data,
    compute_gpiv_from_dataset,
)

### Figure saving

Every figure below is written with the package version appended to the filename,
so you can run the notebook on the old version and the new one and compare the
PNGs directly.

**Where:** `~/My Drive/tcpyVPI_figures/` on your Mac (Colab writes to
`/content/drive/MyDrive/tcpyVPI_figures/`). The repo lives in Dropbox, which
Colab cannot mount, so the figures go to a standalone Drive folder rather than
into the repo. Drive for desktop may take a minute to sync them down, and the
folder does not exist until you run the cell below -- which prints the exact
path it is using.

Set `SAVE_TO_DRIVE = False` to keep everything in `/content` for one session.

The custom figures use **fixed colour limits**, so the two versions are directly
comparable by eye. (The built-in `plot_vpigpiv` panels autoscale, so their
colourbars will differ between runs -- compare those with care.)


In [ ]:
# --- figure saving: every file gets the package version in its name -----------
# Run this notebook once per version and the PNGs sit side by side for comparison.
import os, re, subprocess, tcpyVPI

# WHERE THE FILES GO.
# The repo itself lives in Dropbox, which Colab cannot mount -- so figures go to
# a standalone Google Drive folder instead. That survives a runtime restart,
# which matters because switching package versions cleanly means restarting.
# On your Mac it appears at:  ~/My Drive/tcpyVPI_figures/
SAVE_TO_DRIVE = True
DRIVE_SUBDIR  = "tcpyVPI_figures"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = os.path.join("/content/drive/MyDrive", DRIVE_SUBDIR)
else:
    OUTDIR = DRIVE_SUBDIR          # /content only, erased on disconnect

os.makedirs(OUTDIR, exist_ok=True)

def version_tag():
    """Version string for filenames. Appends the short commit when installed
    from git, so two different commits of the same version cannot collide."""
    tag = tcpyVPI.__version__
    freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
    line = [l for l in freeze.splitlines() if l.lower().startswith("tcpyvpi")]
    if line:
        sha = re.search(r"@([0-9a-f]{7,40})$", line[0].strip())
        if sha:
            tag = f"{tag}+g{sha.group(1)[:7]}"
    return re.sub(r"[^A-Za-z0-9._+-]", "-", tag)

VERSION_TAG = version_tag()
print("figures tagged :", VERSION_TAG)
print("SAVING INTO    :", os.path.abspath(OUTDIR))
if OUTDIR.startswith("/content/drive/MyDrive"):
    print("on your Mac    : ~/My Drive/" + OUTDIR.split("/content/drive/MyDrive/", 1)[1])
else:
    print("NOTE           : /content is erased when the runtime disconnects.")

def savefig(fig, name, dpi=140):
    """Save `fig` as <OUTDIR>/<name>_v<VERSION_TAG>.png"""
    path = os.path.join(OUTDIR, f"{name}_v{VERSION_TAG}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print("saved", os.path.basename(path))
    return path

## 1. Option A - defaults (the published configuration)

`run_vpigpiv` with nothing but a date reproduces Chavas et al. (2025) exactly.
This is what you want unless you are deliberately testing sensitivity.


In [ ]:
year, month = 2022, 9

results = run_vpigpiv(year, month, data_source='monthly', plot=True)
print("\nVariables computed:", list(results.data_vars))

# plot_vpigpiv draws several figures internally; save whatever it opened.
import matplotlib.pyplot as plt
for n, num in enumerate(plt.get_fignums(), start=1):
    savefig(plt.figure(num), f"builtin_panel{n}_{year}{month:02d}")

## 2. Load and compute separately, then make your own plot

Splitting the two steps lets you keep the raw ERA5 fields around and plot
whatever you like.


In [ ]:
ds = load_era5_data(year, month, data_source='monthly')
results = compute_gpiv_from_dataset(ds)

results

Loading ERA5 monthly mean data for 2022-09...
  Loading SSTK...
  Loading SP...
  Loading T...


### Sample plot: potential intensity vs. ventilated potential intensity

The difference between the two panels is the whole point of the ventilated PI.
`PI` is the thermodynamic ceiling. `vPI` is what the storm can actually reach
once shear and mid-level dry air are accounted for, and it cuts off sharply to
zero where the ventilation index exceeds 0.145.


In [ ]:
proj = ccrs.PlateCarree(central_longitude=180)
fig, axes = plt.subplots(3, 1, figsize=(11, 12),
                         subplot_kw={'projection': proj},
                         constrained_layout=True)

panels = [
    ('PI',                results['PI'],                'Potential intensity  [m s$^{-1}$]',        dict(vmin=0, vmax=100, cmap='viridis')),
    ('vPI',               results['vPI'],               'Ventilated PI  [m s$^{-1}$]',              dict(vmin=0, vmax=100, cmap='viridis')),
    ('ventilation_index', results['ventilation_index'], 'Ventilation index  [-]',                   dict(vmin=0, vmax=0.3, cmap='plasma')),
]

for ax, (name, field, label, kw) in zip(axes, panels):
    p = field.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                              add_colorbar=True, cbar_kwargs={'label': label,
                                                              'shrink': 0.85},
                              **kw)
    ax.coastlines(linewidth=0.6)
    ax.add_feature(cfeature.LAND, facecolor='0.85', zorder=2)
    ax.set_extent([-180, 180, -50, 50], crs=ccrs.PlateCarree())
    ax.set_title(f"{name}   ERA5 {year}-{month:02d}")

savefig(fig, f"PI_vPI_VI_{year}{month:02d}")
plt.show()

### The reduction from ventilation

`PI - vPI` is how much intensity the environment takes away. It is largest where
shear is strong and the mid-troposphere is dry -- the subtropics, and the eastern
sides of the basins.


In [ ]:
reduction = (results['PI'] - results['vPI']).where(results['PI'] > 0)

fig, ax = plt.subplots(figsize=(11, 4.5),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)},
                       constrained_layout=True)
reduction.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                          vmin=0, vmax=100, cmap='magma_r',
                          cbar_kwargs={'label': 'PI $-$ vPI  [m s$^{-1}$]', 'shrink': 0.85})
ax.coastlines(linewidth=0.6)
ax.add_feature(cfeature.LAND, facecolor='0.85', zorder=2)
ax.set_extent([-180, 180, -50, 50], crs=ccrs.PlateCarree())
ax.set_title(f"Intensity lost to ventilation   ERA5 {year}-{month:02d}")
savefig(fig, f"PI_minus_vPI_{year}{month:02d}")
plt.show()

## 3. Option B - every configurable input exposed

Since v1.2.0 the choices that used to be hardcoded are keyword arguments. The
dict below lists **all eleven at their default values**, so running this cell
unchanged must give exactly the same answer as Option A. Edit any of them to
test sensitivity.

In [ ]:
PARAMS = dict(
    # --- pressure levels -------------------------------------------------
    shear_p_top   = 200.0,     # hPa, top of the bulk shear layer
    shear_p_bot   = 850.0,     # hPa, bottom of the bulk shear layer
    chi_p_mid     = 600.0,     # hPa, mid-level for the entropy deficit
    vort_level    = 850.0,     # hPa, level of the relative vorticity
    # --- thresholds ------------------------------------------------------
    vort_cap      = 3.7e-5,    # s^-1, cap on absolute vorticity
    VI_max        = 0.145,     # ventilation index above which vPI = 0
    # --- GPIv fit --------------------------------------------------------
    gpiv_exponent = 4.90,      # GPIv = (102.1 * vPI * eta_c) ** exponent
    # --- potential intensity (passed to tcpyPI) --------------------------
    CKCD          = 0.9,       # ratio C_k / C_d
    ascent_flag   = 0,         # 0 = reversible, 1 = pseudo-adiabatic
    diss_flag     = 1,         # 1 = dissipative heating on, 0 = off
    ptop          = 50.0,      # hPa, sounding above this is ignored
)

# NOTE: levels are selected exactly, not by nearest neighbour. If you set a
# level the dataset does not contain you get a KeyError listing what is
# available; interpolate the dataset onto that level first.

# Reuse the ds already loaded in section 2 rather than downloading again -
# same input data, so any difference is due to PARAMS alone.
results_custom = compute_gpiv_from_dataset(ds, verbose=False, **PARAMS)

# (run_vpigpiv forwards these too, if you would rather start from a date:
#  results_custom = run_vpigpiv(year, month, plot=False, verbose=False, **PARAMS))
print("computed with custom PARAMS:", list(results_custom.data_vars))

### Compare Option A against Option B

With `PARAMS` untouched every difference below is exactly zero -- a useful check
that the defaults really are the published configuration. Change something in
`PARAMS`, re-run both cells, and the table shows what it did.

In [ ]:
import pandas as pd

rows = []
for k in ['PI', 'VWS', 'Chi', 'ventilation_index', 'vPI', 'eta_c', 'GPIv']:
    a = results[k].values
    b = results_custom[k].values
    both = np.isfinite(a) & np.isfinite(b)
    denom = np.nanmean(np.abs(a[both])) if both.any() else np.nan
    rows.append({
        'field':        k,
        'default mean': np.nanmean(a),
        'custom mean':  np.nanmean(b),
        'max |diff|':   np.nanmax(np.abs(a[both] - b[both])) if both.any() else np.nan,
        '% change':     100*(np.nanmean(b[both]) - np.nanmean(a[both]))/denom
                        if both.any() and denom else np.nan,
    })
cmp = pd.DataFrame(rows).set_index('field')
identical = np.nanmax(cmp['max |diff|'].values) == 0
print("IDENTICAL to the defaults" if identical
      else "DIFFERENT from the defaults - PARAMS has been modified")
cmp.round(6)

### Map the difference

Only meaningful once you have changed something in `PARAMS`.

In [ ]:
field = 'vPI'          # try 'PI', 'ventilation_index', 'GPIv', ...
diff = results_custom[field] - results[field]

if np.nanmax(np.abs(diff.values)) == 0:
    print(f"No difference in {field}: PARAMS is still at the defaults.")
else:
    lim = float(np.nanpercentile(np.abs(diff.values), 99))
    fig, ax = plt.subplots(figsize=(11, 4.5),
                           subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)},
                           constrained_layout=True)
    diff.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                         vmin=-lim, vmax=lim, cmap='RdBu_r',
                         cbar_kwargs={'label': f'{field}: custom - default', 'shrink': 0.85})
    ax.coastlines(linewidth=0.6)
    ax.add_feature(cfeature.LAND, facecolor='0.85', zorder=2)
    ax.set_extent([-180, 180, -50, 50], crs=ccrs.PlateCarree())
    ax.set_title(f"{field}  custom minus default   ERA5 {year}-{month:02d}")
    savefig(fig, f"diff_{field}_{year}{month:02d}")
    plt.show()

## 4. Sanity check on the values

Quick check that the numbers are physically sensible. Tropical PI in September
should peak somewhere around 80-95 m/s over the warm pool, not above ~110.

Versions of tcpyVPI **up to and including v1.0.1** passed surface pressure to
`tcpyPI` in Pa instead of hPa, and specific humidity where mixing ratio was
expected, which biased PI high by roughly 25%. Both were fixed in v1.1.0. If you
see PI maxima well above 110 m/s here, check which version you installed.


In [ ]:
import csv

rows = []
for name in ['PI', 'vPI', 'ventilation_index', 'Chi', 'VWS', 'GPIv']:
    f = results[name]
    rows.append({'field': name,
                 'min':  float(f.min()),
                 'mean': float(f.mean()),
                 'max':  float(f.max())})
    print(f"{name:18s} min={rows[-1]['min']:10.4f}   "
          f"mean={rows[-1]['mean']:10.4f}   max={rows[-1]['max']:10.4f}")

# Write the same numbers to CSV so two versions can be differenced exactly,
# not just eyeballed.
stats_path = os.path.join(OUTDIR, f"stats_{year}{month:02d}_v{VERSION_TAG}.csv")
with open(stats_path, 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=['field', 'min', 'mean', 'max'])
    w.writeheader(); w.writerows(rows)
print("\nsaved", stats_path)

pi_max = float(results['PI'].max())
print()
if pi_max > 110:
    print(f"WARNING: PI max = {pi_max:.1f} m/s looks too high - are you on a pre-v1.1.0 build?")
else:
    print(f"PI max = {pi_max:.1f} m/s - in the expected range.")

## 5. Collect the output

Everything written this run, ready to compare against another version.

In [ ]:
import glob
print("Files in", os.path.abspath(OUTDIR), "\n")
for f in sorted(glob.glob(os.path.join(OUTDIR, "*"))):
    print(f"  {os.path.getsize(f)/1024:8.1f} KB  {os.path.basename(f)}")

# If you saved to /content instead of Drive, grab them before the runtime dies:
# !zip -qr tcpyVPI_figures.zip "$OUTDIR"
# from google.colab import files; files.download("tcpyVPI_figures.zip")

# Already have two versions in this directory? Difference their stats:
# import pandas as pd
# a = pd.read_csv(f"{OUTDIR}/stats_202209_v1.0.1.csv").set_index("field")
# b = pd.read_csv(f"{OUTDIR}/stats_202209_v1.1.0.csv").set_index("field")
# print((b - a) / a * 100)   # percent change, new vs old
